<a href="https://colab.research.google.com/github/bkekgathetse/setswana-offensive-977/blob/main/notebooks/statistics_reporting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
## Purpose: to show the reliability of your performance metrics across folds.
##
import numpy as np

puoberta = [0.8971, 0.8076, 0.8076, 0.8525, 0.8717]
afroxlmr  = [0.8779, 0.8267, 0.8139, 0.8012, 0.8717]

def bootstrap_ci(scores, n_bootstrap=1000, ci=95):
    boot_means = []
    for _ in range(n_bootstrap):
        sample = np.random.choice(scores, size=len(scores), replace=True)
        boot_means.append(np.mean(sample))
    lower = np.percentile(boot_means, (100 - ci) / 2)
    upper = np.percentile(boot_means, 100 - (100 - ci) / 2)
    return lower, upper

# Example -
PuoBERTa_f1_scores = [0.8971, 0.8076, 0.8076, 0.8525, 0.8717]
ci_lower, ci_upper = bootstrap_ci(PuoBERTa_f1_scores)
print(f"95% CI for PuoBERTa: [{ci_lower:.3f}, {ci_upper:.3f}]")

afroxlmr_f1_scores  = [0.8779, 0.8267, 0.8139, 0.8012, 0.8717]
ci_lower, ci_upper = bootstrap_ci(afroxlmr_f1_scores)
print(f"95% CI for AfroXLM-R: [{ci_lower:.3f}, {ci_upper:.3f}]")

## Use the Wilcoxon test to determine if the performance difference is statistically significant across folds.
## The improvement of PuoBERTa over AfroXLM-R is statistically significant (Wilcoxon signed-rank test, p < 0.05)
from scipy.stats import wilcoxon



stat, p = wilcoxon(puoberta, afroxlmr)
print(f"Wilcoxon test statistic={stat:.3f}, p-value={p:.3f}")




## 📏 3. Effect Sizes
##        For interpretability, compute the magnitude of the improvement (not just significance).

def cohens_d(scores1, scores2):
    diff = np.mean(scores1) - np.mean(scores2)
    pooled_sd = np.sqrt((np.std(scores1, ddof=1)**2 + np.std(scores2, ddof=1)**2) / 2)
    return diff / pooled_sd

d = cohens_d(puoberta, afroxlmr)
print(f"Cohen's d: {d:.2f}")


def cliffs_delta(a, b):
    """
    Compute Cliff's Delta, a non-parametric effect size measure.

    Parameters
    ----------
    a, b : list or np.ndarray
        Two samples (e.g., model scores across folds).

    Returns
    -------
    delta : float
        The Cliff's Delta value in [-1, 1].
    magnitude : str
        Qualitative interpretation: 'negligible', 'small', 'medium', or 'large'.

    Interpretation
    --------------
    delta > 0  → group 'a' tends to have higher values than 'b'
    delta < 0  → group 'b' tends to have higher values than 'a'
    """

    a = np.array(a, dtype=float)
    b = np.array(b, dtype=float)
    n_a, n_b = len(a), len(b)

    # Count pairwise comparisons
    greater = sum(x > y for x in a for y in b)
    less = sum(x < y for x in a for y in b)
    delta = (greater - less) / (n_a * n_b)

    # Interpret magnitude (Romano et al., 2006)
    ad = abs(delta)
    if ad < 0.147:
        magnitude = "negligible"
    elif ad < 0.33:
        magnitude = "small"
    elif ad < 0.474:
        magnitude = "medium"
    else:
        magnitude = "large"

    return delta, magnitude

delta, magnitude = cliffs_delta(puoberta, afroxlmr)
print(f"Cliff's Δ = {delta:.2f} ({magnitude} effect)")




95% CI for PuoBERTa: [0.817, 0.878]
95% CI for AfroXLM-R: [0.811, 0.865]
Wilcoxon test statistic=3.000, p-value=0.625
Cohen's d: 0.24
Cliff's Δ = 0.08 (negligible effect)


In [ ]:
## Purpose: to show the reliability of your performance metrics across folds for SVM, Logistic Regression, and NB.
##
import numpy as np

### TF_IDF baselines F-Macro across folds
svm_TFIDF = [0.8253, 0.8708, 0.8323, 0.8649, 0.8266]
#Fold 1 F1 Macro: 0.8253
#Fold 2 F1 Macro: 0.8708
#Fold 3 F1 Macro: 0.8323
#Fold 4 F1 Macro: 0.8649
#Fold 5 F1 Macro: 0.8266

logistic_TFIDF  = [0.7715, 0.8181, 0.7723, 0.8191, 0.8175]
 #Fold 1: F1 Macro = 0.7715
 #Fold 2: F1 Macro = 0.8181
 #Fold 3: F1 Macro = 0.7723
 #Fold 4: F1 Macro = 0.8191
 #Fold 5: F1 Macro = 0.8175

NB_TFIDF = [0.8333, 0.8782,  0.8077, 0.8269, 0.7817]
#Fold 1 F1 Score: 0.8333
#Fold 2 F1 Score: 0.8782
#Fold 3 F1 Score: 0.8077
#Fold 4 F1 Score: 0.8269
#Fold 5 F1 Score: 0.7817

def bootstrap_ci(scores, n_bootstrap=10000, ci=95):
    boot_means = []
    for _ in range(n_bootstrap):
        sample = np.random.choice(scores, size=len(scores), replace=True)
        boot_means.append(np.mean(sample))
    lower = np.percentile(boot_means, (100 - ci) / 2)
    upper = np.percentile(boot_means, 100 - (100 - ci) / 2)
    return lower, upper

# Example -
ci_lower, ci_upper = bootstrap_ci(svm_TFIDF)
print(f"95% CI for SVM: [{ci_lower:.3f}, {ci_upper:.3f}]")

ci_lower, ci_upper = bootstrap_ci(logistic_TFIDF)
print(f"95% CI for Logistic Regression: [{ci_lower:.3f}, {ci_upper:.3f}]")

ci_lower, ci_upper = bootstrap_ci(NB_TFIDF)
print(f"95% CI for Naive-Bayes: [{ci_lower:.3f}, {ci_upper:.3f}]")


## Use the Wilcoxon test to determine if the performance difference is statistically significant across folds.
## The improvement of PuoBERTa over AfroXLM-R is statistically significant (Wilcoxon signed-rank test, p < 0.05)

# NB vs LR, LR vs SVM, NB vs SVM

from scipy.stats import wilcoxon

# NB vs LR
stat, p = wilcoxon(NB_TFIDF, NB_TFIDF)
print(f"Wilcoxon test statistic: NB vs LR ={stat:.3f}, p-value={p:.3f}")

# LR vs SVM
stat, p = wilcoxon(logistic_TFIDF, svm_TFIDF)
print(f"Wilcoxon test statistic: LR vs SVM ={stat:.3f}, p-value={p:.3f}")

# SVM vs NB
stat, p = wilcoxon(svm_TFIDF, NB_TFIDF)
print(f"Wilcoxon test statistic: SVM vs NB ={stat:.3f}, p-value={p:.3f}")




## 📏 3. Effect Sizes
##        For interpretability, compute the magnitude of the improvement (not just significance).

def cohens_d(scores1, scores2):
    diff = np.mean(scores1) - np.mean(scores2)
    pooled_sd = np.sqrt((np.std(scores1, ddof=1)**2 + np.std(scores2, ddof=1)**2) / 2)
    return diff / pooled_sd

# Cohen's d -->> NB vs LR
d = cohens_d(NB_TFIDF, logistic_TFIDF)
print(f"Cohen's d -->> NB vs LR: {d:.2f}")

# Cohen's d -->>  LR vs SVM
d = cohens_d(logistic_TFIDF, svm_TFIDF)
print(f"Cohen's d -->> LR vs SVM: {d:.2f}")

# Cohen's d -->>  SVM vs NB
d = cohens_d(svm_TFIDF, NB_TFIDF)
print(f"Cohen's d -->> SVM vs NB: {d:.2f}")




def cliffs_delta(a, b):
    """
    Compute Cliff's Delta, a non-parametric effect size measure.

    Parameters
    ----------
    a, b : list or np.ndarray
        Two samples (e.g., model scores across folds).

    Returns
    -------
    delta : float
        The Cliff's Delta value in [-1, 1].
    magnitude : str
        Qualitative interpretation: 'negligible', 'small', 'medium', or 'large'.

    Interpretation
    --------------
    delta > 0  → group 'a' tends to have higher values than 'b'
    delta < 0  → group 'b' tends to have higher values than 'a'
    """

    a = np.array(a, dtype=float)
    b = np.array(b, dtype=float)
    n_a, n_b = len(a), len(b)

    # Count pairwise comparisons
    greater = sum(x > y for x in a for y in b)
    less = sum(x < y for x in a for y in b)
    delta = (greater - less) / (n_a * n_b)

    # Interpret magnitude (Romano et al., 2006)
    ad = abs(delta)
    if ad < 0.147:
        magnitude = "negligible"
    elif ad < 0.33:
        magnitude = "small"
    elif ad < 0.474:
        magnitude = "medium"
    else:
        magnitude = "large"

    return delta, magnitude

# NB vs LR
delta, magnitude = cliffs_delta(NB_TFIDF, logistic_TFIDF)
print(f"Cliff's Δ -->>  NB vs LR = {delta:.2f} ({magnitude} effect)")

# LR vs SVM
delta, magnitude = cliffs_delta(logistic_TFIDF, svm_TFIDF)
print(f"Cliff's Δ -->> LR vs SVM = {delta:.2f} ({magnitude} effect)")

# SVM vs NB
delta, magnitude = cliffs_delta(svm_TFIDF, NB_TFIDF)
print(f"Cliff's Δ -->> NB vs LR = {delta:.2f} ({magnitude} effect)")




95% CI for SVM: [0.827, 0.861]
95% CI for Logistic Regression: [0.781, 0.818]
95% CI for Naive-Bayes: [0.800, 0.855]
Wilcoxon test statistic: NB vs LR =0.000, p-value=1.000
Wilcoxon test statistic: LR vs SVM =0.000, p-value=0.062
Wilcoxon test statistic: SVM vs NB =3.000, p-value=0.312
Cohen's d -->> NB vs LR: 0.84
Cohen's d -->> LR vs SVM: -1.86
Cohen's d -->> SVM vs NB: 0.62
Cliff's Δ -->>  NB vs LR = 0.52 (large effect)
Cliff's Δ -->> LR vs SVM = -1.00 (large effect)
Cliff's Δ -->> NB vs LR = 0.20 (small effect)
